In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\data\sampled_data\sampled_data.csv")

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# ---------------------------------------------------------
# 1. MA'LUMOTLARNI TAYYORLASH (Custom Dataset)
# ---------------------------------------------------------

# PyTorch ma'lumotlarni DataLoader orqali qabul qiladi. 
# Buning uchun o'zimizning Dataset klassimizni yozamiz:
class CarDataset(Dataset):
    def __init__(self, X, y_range, y_price):
        # Ma'lumotlarni PyTorch Tensor (massiv) formatiga o'tkazamiz
        self.X = torch.tensor(X, dtype=torch.float32)
        # Multi-class classification uchun targetlar butun son (long/int64) bo'lishi shart
        self.y_range = torch.tensor(y_range.values, dtype=torch.long)
        self.y_price = torch.tensor(y_price.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_range[idx], self.y_price[idx]


X = df.drop(columns=['Range_km_level', 'Price_level']).values
y1 = df['Range_km_level']
y2 = df['Price_level']

X_train, X_test, y1_train, y1_test, y2_train, y2_test = train_test_split(
    X, y1, y2, test_size=0.2, random_state=42
)

# Deep Learning uchun standartlashtirish muhim
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Dataset va DataLoader yaratish
train_dataset = CarDataset(X_train_scaled, y1_train, y2_train)
test_dataset = CarDataset(X_test_scaled, y1_test, y2_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ---------------------------------------------------------
# 2. MODEL ARXITEKTURASI (nn.Module)
# ---------------------------------------------------------

class MultiOutputNN(nn.Module):
    def __init__(self, input_dim, num_classes_range, num_classes_price):
        super(MultiOutputNN, self).__init__()
        
        # Umumiy qatlamlar (Shared Layers)
        self.shared_layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        
        # 1-shox: Range_km_level uchun
        self.range_branch = nn.Linear(64, num_classes_range)
        
        # 2-shox: Price_level uchun
        self.price_branch = nn.Linear(64, num_classes_price)

    def forward(self, x):
        # Ma'lumot avval umumiy qatlamdan o'tadi
        shared_features = self.shared_layers(x)
        
        # Keyin ikkiga ajraladi
        out_range = self.range_branch(shared_features)
        out_price = self.price_branch(shared_features)
        
        return out_range, out_price

# Modelni ishga tushirish
input_dim = X_train_scaled.shape[1]
num_classes_range = len(y1.unique())
num_classes_price = len(y2.unique())

model = MultiOutputNN(input_dim, num_classes_range, num_classes_price)

# ---------------------------------------------------------
# 3. LOSS FUNKSIYALAR VA OPTIMIZER
# ---------------------------------------------------------

# PyTorch'da CrossEntropyLoss ichida avtomatik tarzda Softmax qilingan bo'ladi!
# Shuning uchun model oxiriga softmax yozmadik.
criterion_range = nn.CrossEntropyLoss()
criterion_price = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------------------------------------
# 4. O'QITISH (TRAINING LOOP)
# ---------------------------------------------------------

epochs = 50

print("PyTorch modeli o'qitilmoqda...\n")

for epoch in range(epochs):
    model.train() # Modelni o'qitish rejimiga o'tkazish
    running_loss = 0.0
    
    for batch_X, batch_y_range, batch_y_price in train_loader:
        # Gradientlarni tozalash
        optimizer.zero_grad()
        
        # Modelga bashorat qildirish
        pred_range, pred_price = model(batch_X)
        
        # Xatoliklarni (Loss) alohida hisoblash
        loss_range = criterion_range(pred_range, batch_y_range)
        loss_price = criterion_price(pred_price, batch_y_price)
        
        # Ikkala xatolikni qo'shib, umumiy xatolikni topish
        total_loss = loss_range + loss_price
        
        # Xatoni orqaga qaytarish (Backpropagation)
        total_loss.backward()
        
        # Og'irliklarni yangilash
        optimizer.step()
        
        running_loss += total_loss.item()
        
    # Har 10 epoch'da natijani ekranga chiqarish
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Umumiy Loss: {running_loss/len(train_loader):.4f}")

print("\nO'qitish yakunlandi!")

PyTorch modeli o'qitilmoqda...

Epoch [10/50], Umumiy Loss: 0.4026
Epoch [20/50], Umumiy Loss: 0.2776
Epoch [30/50], Umumiy Loss: 0.2311
Epoch [40/50], Umumiy Loss: 0.2011
Epoch [50/50], Umumiy Loss: 0.1620

O'qitish yakunlandi!
